# SentencePiece, step by step

SentencePiece trains a tokenizer directly from raw text. It supports both **Unigram** and **BPE** models.

In [2]:
from pathlib import Path
import sys
import tempfile

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "SentencePiece" / "code").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from inside the tokenizer_implementations repository")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from SentencePiece.code.tokenizer import SentencePieceTokenizer

print("Python kernel:", sys.executable)
print("Project root:", project_root)

Python kernel: d:\project\.venv\Scripts\python.exe
Project root: d:\project\tokenizer_implementations


## 1. Start with raw sentences

There is no need to split words first. SentencePiece learns from raw strings and preserves word boundaries with the `▁` marker.

In [3]:
corpus = [
    "low lower lowest",
    "new newer newest",
    "wide wider widest",
    "the tokenizer learns subword pieces",
    "a tokenizer can encode and decode text",
    "नमस्ते दुनिया",
    "hello world",
]
text = "a newer tokenizer"
print("Training sentences:", len(corpus))
print("Example text:", text)

Training sentences: 7
Example text: a newer tokenizer


## 2. Train both model types

Unigram selects useful pieces from a candidate vocabulary. BPE starts with small symbols and repeatedly merges frequent neighbors. Both are saved in SentencePiece's `.model` and `.vocab` formats.

In [4]:
temporary_directory = tempfile.TemporaryDirectory()
model_directory = Path(temporary_directory.name)

unigram = SentencePieceTokenizer.train(
    corpus,
    model_prefix=model_directory / "unigram",
    model_type="unigram",
    vocab_size=64,
    hard_vocab_limit=False,
)
bpe = SentencePieceTokenizer.train(
    corpus,
    model_prefix=model_directory / "bpe",
    model_type="bpe",
    vocab_size=64,
    hard_vocab_limit=False,
)

print("Unigram vocabulary:", unigram.vocabulary_size)
print("BPE vocabulary:", bpe.vocabulary_size)

Unigram vocabulary: 56
BPE vocabulary: 64


## 3. Encode into pieces and IDs

Pieces are readable subwords. IDs are their integer positions in the model vocabulary.

In [5]:
for name, tokenizer in [("Unigram", unigram), ("BPE", bpe)]:
    pieces = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)
    print(f"{name} pieces:", pieces)
    print(f"{name} IDs:   ", token_ids)
    print(f"{name} decode:", tokenizer.decode(token_ids))
    print()

Unigram pieces: ['▁a', '▁new', 'er', '▁tokenizer']
Unigram IDs:    [45, 9, 22, 17]
Unigram decode: a newer tokenizer

BPE pieces: ['▁', 'a', '▁new', 'er', '▁tokenizer']
BPE IDs:    [33, 44, 16, 5, 29]
BPE decode: a newer tokenizer



## 4. Inspect whitespace and special tokens

A piece beginning with `▁` starts after a space or at the beginning of the sentence. The model also reserves IDs for unknown, beginning-of-sentence, end-of-sentence, and padding tokens.

In [6]:
pieces = unigram.tokenize("hello world")
print("Pieces:", pieces)
print("Whitespace-marked pieces:", [piece for piece in pieces if piece.startswith("▁")])
print("UNK ID:", unigram.processor.unk_id())
print("BOS ID:", unigram.processor.bos_id())
print("EOS ID:", unigram.processor.eos_id())
print("PAD ID:", unigram.processor.pad_id())
print("With BOS and EOS:", unigram.encode(text, add_bos=True, add_eos=True))

Pieces: ['▁', 'he', 'l', 'lo', '▁', 'wor', 'l', 'd']
Whitespace-marked pieces: ['▁', '▁']
UNK ID: 0
BOS ID: 1
EOS ID: 2
PAD ID: 3
With BOS and EOS: [1, 45, 9, 22, 17, 2]


## 5. Sample alternative Unigram segmentations

Subword regularization samples other valid segmentations. This is mainly useful while preparing model-training data. BPE normally uses one deterministic segmentation.

In [7]:
for _ in range(5):
    print(unigram.sample(text, alpha=0.5, nbest_size=-1))

['▁a', '▁new', 'er', '▁tokenizer']
['▁a', '▁newe', 'r', '▁tokenizer']
['▁a', '▁new', 'er', '▁tokenizer']
['▁a', '▁new', 'er', '▁tokenizer']
['▁a', '▁newe', 'r', '▁tokenizer']


## 6. Inspect saved artifacts

The binary `.model` file is loaded for tokenization. The text `.vocab` file is useful for inspecting learned pieces and their scores.

In [8]:
print(sorted(path.name for path in model_directory.iterdir()))
print("First vocabulary entries:")
print("\n".join((model_directory / "unigram.vocab").read_text(encoding="utf-8").splitlines()[:10]))

['bpe.model', 'bpe.vocab', 'unigram.model', 'unigram.vocab']
First vocabulary entries:
<unk>	0
<s>	0
</s>	0
<pad>	0
▁	-2.02744
d	-3.35952
▁wide	-3.4002
r	-3.48279
▁low	-3.65081
▁new	-3.65658
